
#Projectile Motion using SMT solvers

The kinematic equations describe a projectile completely, but using them means picking
the right one for the unknown you want and rearranging it. In this notebook we will look
at SMT solvers and give Z3 all of the equations at once instead, then ask it for
whichever quantity we are missing. We will also need sines and cosines, which Z3 has no
built-in notion of, so we will see one way of giving it those too.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, Sin, Cos, draw_projectile
import math
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving real numbers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Real('x') # declairing that x is a real number named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## Projectile Motion
Projectile motion is defined as the motion of an object that is projected (thrown or launched) into air with only the force of gravity acting upon it once launched.  
Typically we will ignore air resistance and other effects (such as spinning) when studying projectile motion.

## Splitting 2D motion into $x$ and $y$ components
We can take 2-dimensional projectile motion and split it into two parts:
the horizontal motion, and the vertical motion.  
In our case, since projectile motion only has the force of gravity acting on the projectile, we can say two things:

1. The acceleration in the $y$ direction is equal to the acceleration due to gravity.
2. The acceleration in the $x$ direction is equal to 0

Let's initialize our solver and add those two constraints first.

In [ ]:
solver = Solver()

g = Real("g")
a_x, a_y = Reals("a_x a_y")

solver.add(g == 9.81) # 9.81 m/s^2
solver.add(a_x == 0)  # acceleration in the x direction
solver.add(a_y == -g) # acceleration in the y direction

## Horizontal Motion
We can use the kinematic equations to model the horizontal motion of a projectile.
$$
v_t = v_0 + a t \\
x_t = x_0 + v_0 t + \frac{1}{2} a t^2 \\
x_t = x_0 + \frac{v_t + v_0}{2} t \\
v_t^2 = v_0^2 + 2 a (x_t - x_0)
$$
Since $a_x = 0$, we can simplify 
$$
v_{tx} = v_{0x} \\
x_{tx} = x_{0x} + v_{tx} t \\
$$
Let's add these constraints to our solver.

In [ ]:
v_0x, v_tx, x_0, x_t, t = Reals("v_0x v_tx x_0 x_t t")

solver.add(v_tx == v_0x)
solver.add(x_t == x_0 + v_tx*t)

## Vertical Motion
For vertical motion we have the following:
$$
v_{ty} = v_{0y} - g t \\
y_t = y_0 + v_{0y} t - \frac{1}{2} g t^2 \\
y_t = y_0 + \frac{v_{ty} + v_{0y}}{2} t \\
v_{ty}^2 = v_{0y}^2 - 2 g (y_t - y_0)
$$
Let's add these constraints to our solver.

Three of the four are written for you. **Replace the marked line** with the equation
above that gives the vertical position $y_t$ in terms of the initial position, the
initial vertical velocity, and the time spent falling.

In [ ]:
v_0y, v_ty, y_0, y_t = Reals("v_0y v_ty y_0 y_t")

solver.add(v_ty == v_0y - g*t)
solver.add(y_t == y_0) # REPLACE THIS LINE
solver.add(y_t == y_0 + (v_ty + v_0y)/RealVal(2)*t)
solver.add(v_ty**2 == v_0y**2 - 2*g*(y_t - y_0))

Now we can test our solver by giving it some initial constraints.  
Consider the following problem:  
A cannonball is launched from a 20 meter high building at an angle of 45 degrees, with an initial velocity of 20 meters per second.  
How long does it take for the cannonball to land?

First we need to split the initial velocity into $x$ and $y$ components.  
We can do that using trigonometry, and in fact, we will use Z3 to do that for us using the following relations:
$$
v_x = v \cos(\theta) \\
v_y = v \sin(\theta)
$$

Z3 has no built-in sine or cosine, so `Sin` and `Cos` work differently from the
functions we have been adding so far: each one takes an angle, a second variable, and
the solver, and constrains that second variable to equal the sine or cosine of the
angle.

The horizontal component is written for you. **Replace the marked line** with the
vertical one.

In [ ]:
th_0, cos_th0, sin_th0 = Reals("th_0 cos_th0 sin_th0")
v_0 = Real("v_0")

Cos(th_0, cos_th0, solver)
Sin(th_0, sin_th0, solver)

solver.add(v_0x == v_0*cos_th0)
solver.add(v_0y == v_0) # REPLACE THIS LINE

Now we can get back to solving our original problem.  
From the problem, we can gather:
$$
v_0 = 20 \\
\theta_0 = 45 \\
y_0 = 20 \\
x_0 = 0 \\
$$
And we want to know the time $t$ when $y_t = 0$.

In [ ]:
solver.add(v_0 == 20)
solver.add(th_0 == math.radians(45)) # need to convert degrees to radians
solver.add(y_0 == 20)
solver.add(x_0 == 0)

solver.add(y_t == 0)

print(solver.check())
print(solver.model())
print(f"t (time) = {solver.model()[t]}")

To make things easier (particularly for graphing) we can wrap a class around the constraints we added to the solver.

In [ ]:
class ProjectileMotion:
    def __init__(self, solver = Solver()):
        self.solver = solver
        self.trig_param = 10

        self.g, self.t = Reals("g t")

        self.y_0, self.x_0 = Reals("y_0 x_0") # initial y position, initial x position
        self.y, self.x = Reals("y x") # y position, x position

        self.v_0, self.v_y0, self.v_x0 = Reals("v_0 v_y0 v_x0") # initial velocities
        self.th_0, self.cos_th0, self.sin_th0 = Reals("th_0 cos_th0 sin_th0") # initial angle in radians

        self.v, self.v_y, self.v_x = Reals("v v_y v_x") # velocities
        self.th, self.cos_th, self.sin_th = Reals("th cos_th sin_th") # angle in radians

        self.solver.add(self.g == 9.81)

        self.solver.add(self.x == self.x_0 + self.v_x*self.t)
        self.solver.add(self.y == self.y_0 + (self.v_y0 + self.v_y)*self.t/RealVal(2))
        self.solver.add(self.y == self.y_0 + self.v_y0*self.t - (self.g*self.t**2)/RealVal(2))

        Cos(self.th_0, self.cos_th0, self.solver, n = self.trig_param)
        Sin(self.th_0, self.sin_th0, self.solver, n = self.trig_param)
        self.solver.add(self.v_x0 == self.v_0 * self.cos_th0)
        self.solver.add(self.v_y0 == self.v_0 * self.sin_th0)

        Cos(self.th, self.cos_th, self.solver, n = self.trig_param)
        Sin(self.th, self.sin_th, self.solver, n = self.trig_param)
        self.solver.add(self.v_x == self.v * self.cos_th)
        self.solver.add(self.v_y == self.v * self.sin_th)
        self.solver.add(self.v_x == self.v_x0)
        self.solver.add(self.v_y == self.v_y0 - self.g*self.t)

    def get_value_from_model(self, variable):
        if self.solver.check() == unsat:
            return None
        else:
            return self.solver.model()[variable].as_decimal(6)

We can then use this class to easily solve any projectile motion problem we may come across.  

For example:  
A soccer ball is kicked into the air with an initial velocity of 35 meters per second (wow!) at 30 degrees above the horizontal.  
What is the maximum height the soccer ball reaches? At what time? How far did it travel horizontally at that point?

Note that a projectile reaches its highest point at the exact time that its vertical velocity $v_y = 0$.

The initial conditions are written for you. **Replace the marked line** with the
constraint that holds at the highest point of the flight.

In [ ]:
pm = ProjectileMotion()

# initial values
pm.solver.add(pm.x_0 == 0)
pm.solver.add(pm.y_0 == 0)
pm.solver.add(pm.v_0 == 35)
pm.solver.add(pm.th_0 == math.radians(30))
pm.solver.add(pm.v_y0 >= 0)
pm.solver.add(pm.v_x0 >= 0)

# at the apex
pm.solver.add(pm.v_y >= 0) # REPLACE THIS LINE

print(pm.solver.check())
print(pm.solver.model())

print("\nat the apex:")
print(f"y (height) = {pm.get_value_from_model(pm.y)}")
print(f"t (time) = {pm.get_value_from_model(pm.t)}")
print(f"x (horizontal displacement) = {pm.get_value_from_model(pm.x)}")

draw_projectile(pm)


###Congratulations! You just used an SMT solver to solve projectile motion problems!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm